In [1]:
#import libraries
import math
import pandas as pd
import numpy as np
import time
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import datetime
import numpy as np
import pandas as pd
import jdatetime
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import t as t_dis
from statsmodels.stats.proportion import proportion_confint
from lifelines import WeibullFitter,\
                      ExponentialFitter,\
                      LogNormalFitter,\
                      LogLogisticFitter

from plotnine import *

c:\Users\x\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
# code for calculating distance
import math
import pandas as pd
import numpy as np
import time
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
def calculate_distance(lat1, lon1, lat2, lon2):
    # Radius of the Earth in meters
    earth_radius = 6371000

    # Convert degrees to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    # Calculate the differences between the latitudes and longitudes
    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad

    # Haversine formula
    a = math.sin(delta_lat / 2) ** 2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(delta_lon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Calculate the distance
    distance = earth_radius * c

    if distance > 150:
        return False
    else:
        return distance

In [ ]:
#merging accident points with telematics df
import pandas as pd
alarm_list = ['01','02','03','04','05','06','07','08','09','10','11','12']
ala = pd.DataFrame()
for i in alarm_list:
    ala1 = pd.read_excel(r"..................................\data\alarm2020{x}.xlsx".format(x=i))
    ala = pd.concat([ala,ala1], axis = 0, ignore_index= True)

beh = pd.DataFrame()
for i in alarm_list:
    beh1 = pd.read_excel(r"..................................\beha2020{x}.xlsx".format(x=i))
    beh = pd.concat([beh,beh1], axis = 0, ignore_index= True)
    
total = pd.read_csv(r"......................................\totaldata.csv")
total.drop(columns = 'Unnamed: 0', inplace = True)
#TYPE 0-4 -->beh & 5-6 ---> ala
#hursh_deceleration=sum(number.0)
#hursh_acceleration=sum(number.1)
#hursh_turning=sum(number.2)
#high_revolution=sum(number.3)
#engine_speed=sum(number.4)
#over_speed=sum(number.5)
#fatigue_alarm=sum(number.6)

accident_points = pd.read_csv(r"..........................\accident_points.csv")
def lat_long_tuple(lat,lng):
    lat_long = (lat, lng)
    return lat_long

accident_points['lat_long'] = accident_points.apply(lambda x: lat_long_tuple(x['lat'],x['lng']), axis = 1)
accients_lat_lng_list = list(accident_points['lat_long'])    


def serious_road(lat,lng):
    distance_list = list()
    for i in accients_lat_lng_list:
        distance_list.append(calculate_distance(lat, lng, i[0], i[1]))
    if any(distance_list) == False:
        return False
    else:
        dist_serie = pd.Series(distance_list)
        dist_serie.replace({False:np.nan}, inplace = True)
        min_index = dist_serie.index[dist_serie == dist_serie.min()][0]
        return [distance_list[min_index],accients_lat_lng_list[min_index]]
    
total['situation'] = total.apply(lambda x: serious_road(x['LAT'],x['LON']), axis = 1)
total['danger'] = total['situation'].apply(lambda x: False if x == False else True)
total.to_pickle(r"...............................................\AG_totaldata_danger.pkl")

In [ ]:
tel = pd.read_pickle(r"..........................................\AG_totaldata_danger.pkl")
accident_points = pd.read_csv(r"..................................\accident_points.csv")

gdf = gpd.read_file(r"C:\Users\x\Desktop\map\E14.shp")
# Define the CRS of your data points (e.g., WGS 84)
data_points_crs = 'EPSG:4326'

# Reproject the country boundaries to match the CRS of your data points
transformer = Transformer.from_crs(gdf.crs, data_points_crs, always_xy=True)
gdf = gdf.to_crs(data_points_crs)

def province_finder(lat,long):
    point = Point(long, lat)
    countries_containing_point = gdf[gdf['geometry'].contains(point)]
    if not countries_containing_point.empty:
        country_name = countries_containing_point.iloc[0]['PROVINCE']  # Replace with your column name
        return country_name
    else:
        return False
    
accident_points['province'] = accident_points.apply(lambda x: province_finder(x['lat'],x['lng']), axis = 1)
#accident_points.at[accident_points[accident_points['province'] == False].index[0], 'province'] = 'Azerbaijan, East'


In [ ]:
def check_province(situ):
    if situ == False:
        return False
    else:
        df = accident_points[(accident_points['lat'] == situ[1][0]) & (accident_points['lng'] == situ[1][1])].copy()
        return df['province'].iloc[0]

tel['province_2'] = tel['situation'].apply(lambda x: check_province(situ))

def check_two_provinces(pro1, pro2):
    if pro2 == False:
        return False
    else:
        if pro1 == pro2:
            return True
        else:
            return 'check'


tel['ckecker'] = tel.apply(lambda x: check_two_provinces(x['province_1'],x['province_2']), axis = 1)
tel.drop(columns= ['province_2', 'ckecker'], inplace= True)
tel.rename(columns= {'province_1': 'province'}, inplace = True)
tel.to_pickle(r"...........................................\AG_totaldata_danger.pkl")

In [ ]:
#adding weather information
from datetime import date, datetime, timedelta
import matplotlib.pyplot as plt
from meteostat import Point, Daily, Hourly
import pandas as pd
import numpy as np

def weather_finder(dates, lat, lon):
    start = dates.date() 
    start = datetime.combine(start, datetime.min.time())
    end = start + timedelta(days=1)

    # Create Point for location
    location = Point(lat, lon)

    # Get data
    data = Hourly(location, start, end)
    data = data.fetch()
    if data.shape[0] > 0:
        data.reset_index(inplace = True)
        data['time_differ'] = data['time'] - dates
        data['time_differ'] = data['time_differ'].apply(lambda x: abs(x.total_seconds()))
        seconds_list = sorted(list(data['time_differ']))
        checker = False
        i = 0
        answer = False
        while (checker == False) & (i <= data.shape[0]):
            result = data[data['time_differ'] == seconds_list[i]].copy()
            result = result[[ 'temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']].copy()
            if all(np.isnan(value) if isinstance(value, float) else False for value in dict(result.iloc[0]).values()):
                i += 1
            else:
                return dict(result.iloc[0])
                checker = True
                answer = True
        if answer == False:
            return np.nan
    else:
        return np.nan

tel['weather_information'] = tel.apply(lambda x: weather_finder(x['time'],x['LAT'],x['LON']), axis = 1)

for i in ['temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'coco']:
    tel[i] = tel['weather_information'].apply(lambda x: np.nan if type(x) != dict else x[i])

#after checking data
tel.drop(columns = ['prcp', 'snow', 'wpgt', 'tsun'], inplace= True)

tel.to_pickle(r"............................................\tel_cleaned_AG.pkl")

In [ ]:
#import libraries
import math
import pandas as pd
import numpy as np
import time
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import datetime

#LOADING TEL
tel = pd.read_pickle(r".....................\tel_cleaned_AG.pkl")
roads = pd.read_csv(r".......................\merged data.csv")
final_roads = pd.read_pickle(r".................\final_roads_AG.pkl")
road_merge = final_roads[['road','type']].copy()
roads = roads[['ID', 'TER_ID','road']].copy()
tel = tel.merge(roads, on = ['ID', 'TER_ID'], how = 'left', validate= '1:m')
tel = tel.merge(road_merge, on = ['road'], how = 'inner', validate= 'm:1')
tel.drop(tel.index[tel.road.isin([261, 1084, 1910])], inplace = True)

def road_changer(road, types):
    if road in [3905,2595,5065,5996]:
        return 'primary'
    elif road in [6386,6215,8010,3490, 1666, 4661]:
        return 'trunk'
    elif road in [2455]:
        return 'motorway'
    else:
        return types
    
tel['type'] = tel.apply(lambda x: road_changer(x['road'], x['type']), axis = 1)
tel['type'].replace({'trunk_link':'trunk',
'primary_link': 'primary', 'motorway_junction': 'motorway', 'secondary_link':'secondary',
'motorway_link': 'motorway', 'service':'residential', 'services':'residential', 'living_street':'residential',
'rest_area': 'trunk', 'unclassified':'minor roads', }, inplace = True)

tel.drop(columns = ['situation',  'weather_information'], inplace= True)
tel.rename(columns = {'DATES': 'month', 'Type':'error_type', 'Day_name':'day_name', 'type':'road_type'},inplace = True)

roads = pd.read_csv(r"...........................................\merged data.csv")
import geopandas as gpd
gdf = gpd.read_file(r"...........................................\Roads-Iran.shp")


# Assuming you have a GeoDataFrame named 'gdf' with the provided data and CRS information

# Set the CRS for the GeoDataFrame (assuming it's not already set)
gdf.crs = 'PROJCS["Custom",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],TOWGS84[0,0,0,0,0,0,0],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.01745329251994328,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["standard_parallel_1",33.878398],PARAMETER["standard_parallel_2",35.000682],PARAMETER["latitude_of_origin",34.5],PARAMETER["central_meridian",49],PARAMETER["false_easting",400000],PARAMETER["false_northing",400000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["X",EAST],AXIS["Y",NORTH],AUTHORITY["EPSG","32640"]]'

# Calculate road lengths and add them to a new column 'road_length'
gdf['road_length'] = gdf['geometry'].length
road_length_df = gdf[['RECNO', 'road_length']].copy()

roads = roads.merge(road_length_df, on = 'RECNO', how = 'left', validate = 'm:1')
roads = roads[['road', 'road_length']].copy()
roads.drop_duplicates(subset= ['road'], inplace= True)
tel = tel.merge(roads, on = 'road', how = 'left', validate = 'm:1')
tel['coco'].replace({0.0:np.nan}, inplace = True)
tel['rhum'] = tel['rhum'] .apply(lambda x: x if x <= 100 else np.nan)

def modify_coco(code):
    if code == 2:
        return 1
    elif code == 4:
        return 3
    elif code in [8, 17, 18, 9, 10]:
        return 7
    elif code in [27, 26]:
        return 25
    elif code in [15, 21, 16, 22, 24, 12, 19, 13]:
        return 14
    else:
        return code

tel['coco'] = tel['coco'].apply(modify_coco)
tel['coco'].replace({1:'Clear', 5:'Foggy', 3:'Cloudy', 7:'Rainy', 25: 'Storm', 14:'Snowy'}, inplace = True)

# Weather condition prediction using HistGradientBoostingClassifier

In [ ]:
tel['month'] = tel['month'].apply(lambda x: str(x))
tel['hour'] = tel['hour'].apply(lambda x: str(x))
tel['day/night'] = tel['day/night'].replace({'D':'1','N':'2', 'T':'3'})
train_df = tel[['month', 'LAT', 'LON', 'temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres', 'day/night', 'hour', 'coco']].copy()
train_df.dropna(subset= 'coco', inplace= True)
train_df.reset_index(inplace= True)
train_df.drop(columns= 'index', inplace = True)

seed = 2019
np.random.seed(seed)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(train_df[['temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres']])

X_final = pd.DataFrame(
    data= X_numeric_scaled,
    columns=['temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres']
)

X_final = pd.concat([X_final, train_df[['LAT', 'LON', 'month','day/night', 'hour']]], axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X_final, train_df['coco'], test_size=0.2, random_state=seed)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# Create the classifier
clf = HistGradientBoostingClassifier(random_state=seed, categorical_features =[False, False, False, False, False, False, False, False, True,True, True])

# Define the parameter grid to search over
param_grid = {
    'max_iter': [100, 200, 300],
    'max_depth': [15, 30, 50],  # Lower values can help reduce overfitting.
    'l2_regularization': [0.0, 0.1, 0.2],  # Regularization strength.
    'learning_rate': [0.1, 0.01],  # Learning rate controls the step size during gradient boosting.
}

hgb_grid = GridSearchCV(clf, param_grid, n_jobs=5,  cv=5, verbose=2, refit=True, scoring= 'f1_weighted')
hgb_grid.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

y_pred = hgb_grid.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
confusion = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Classification Report:\n", report)
print("Confusion Matrix:\n", confusion)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
clf = HistGradientBoostingClassifier(random_state=seed, categorical_features=[False, False, False, False, False, False, False, False,True, True, True],
                                    learning_rate = 0.1, l2_regularization=0.1, max_depth=15,  max_iter=300)
clf.fit(X_train, y_train)  

tel.reset_index(inplace = True)
predict_df = tel[['index', 'month', 'LAT', 'LON', 'temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres', 'day/night', 'hour', 'coco']].copy()
predict_df = predict_df[predict_df['coco'].isna() == True].copy()

def weather_missings_count(row):
    nan_count = row[['temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres']].isna().sum()
    return nan_count 
predict_df['weathers_missings_count'] = predict_df.apply(weather_missings_count, axis=1)

predict_dff = predict_df[predict_df['weathers_missings_count'] < 3].copy()
columns_to_scale = ['temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres']
scaler = StandardScaler()
predict_dff[columns_to_scale] = scaler.fit_transform(predict_dff[columns_to_scale])

predict_dff['coco_pred'] = clf.predict(predict_dff[['temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres', 'LAT', 'LON', 'month','day/night', 'hour']])

tel = tel.merge(predict_dff[['index','coco_pred']], on = 'index', how = 'left', validate = '1:1')

def combine_values(row):
    if not pd.isna(row['coco']):
        return row['coco']
    elif not pd.isna(row['coco_pred']):
        return row['coco_pred']
    else:
        return np.nan

# Create a new column 'final' by applying the function
tel['final_coco'] = tel.apply(combine_values, axis=1)
tel.drop(columns= ['coco','coco_pred'], inplace= True)

tel.to_pickle(r".............................\tel_predicted_coco.pkl")

# Feature Selection

In [ ]:
#import libraries
import math
import pandas as pd
import numpy as np
import time
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

predicted_tel = pd.read_pickle(r"............................................\tel_predicted_coco.pkl")

In [ ]:
rf_df = predicted_tel[['season','month', 'error_type', 'province', 'final_coco','day/night', 'day_name', 
                        'day_type', 'hour', 'road_type','temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres', 'danger']].copy()

categorical_columns = ['season','month', 'error_type','province','day/night', 'day_name', 'day_type', 'hour', 'road_type']

for i in categorical_columns:
    rf_df[i] = rf_df[i].apply(lambda x: str(x))

scaler = StandardScaler()
columns_to_scale = ['temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres']
scaler = StandardScaler()
rf_df[columns_to_scale] = scaler.fit_transform(rf_df[columns_to_scale])

for i in ['temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres']:
    rf_df[i] = rf_df[i] + abs(rf_df[i].min())

rf_df['final_coco'].fillna('Unknown', inplace = True)
rf_df.fillna(-1, inplace = True)

without_missing_feature_df = rf_df[(rf_df['rhum'] >= 0) & (rf_df['dwpt'] >= 0) & (rf_df['wdir'] >= 0) &
         (rf_df['temp'] >= 0) & (rf_df['wspd'] >= 0) & (rf_df['pres'] >= 0) & (rf_df['final_coco'] != 'Unknown')].copy()
nonmiss_X_train, nonmiss_X_test, nonmiss_y_train, nonmiss_y_test = train_test_split(without_missing_feature_df.drop(columns='danger'), without_missing_feature_df['danger'], test_size=0.3, random_state=2019) 


In [ ]:
without_missing_feature_df = rf_df[(rf_df['rhum'] >= 0) & (rf_df['dwpt'] >= 0) & (rf_df['wdir'] >= 0) &
         (rf_df['temp'] >= 0) & (rf_df['wspd'] >= 0) & (rf_df['pres'] >= 0) & (rf_df['final_coco'] != 'Unknown')].copy()
nonmiss_X_train, nonmiss_X_test, nonmiss_y_train, nonmiss_y_test = train_test_split(without_missing_feature_df.drop(columns='danger'), without_missing_feature_df['danger'], test_size=0.3, random_state=2019) 

#feature selection for all available features (all features but na rows dropped)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from imblearn.ensemble import BalancedRandomForestClassifier

#categorical_columns = ['month', 'error_type','province','final_coco','day/night', 'day_name', 'day_type', 'hour', 'road_type']
categorical_columns = ['season', 'month','error_type','province','final_coco','day/night', 'day_name', 'day_type', 'hour','road_type']
numerical_columns = ['temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres']

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",unknown_value=-1, encoded_missing_value=-1)
numerical_pipe = SimpleImputer(strategy='constant', fill_value= -1)

preprocessing = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_pipe, numerical_columns),
    ],
    verbose_feature_names_out=False,
)


clf = Pipeline(
    [
        ("preprocess", preprocessing),
        ("classifier", BalancedRandomForestClassifier(n_estimators = 1000, sampling_strategy="auto", replacement=True, max_depth=100, random_state=0)),
    ]
)

clf.fit(nonmiss_X_train, nonmiss_y_train)

In [ ]:
print(f"RF train accuracy: {clf.score(nonmiss_X_train, nonmiss_y_train):.3f}")
print(f"RF test accuracy: {clf.score(nonmiss_X_test, nonmiss_y_test):.3f}")

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

nonmiss_y_pred = clf.predict(nonmiss_X_test)

# Evaluate the model's performance
accuracy = accuracy_score(nonmiss_y_test, nonmiss_y_pred)
report = classification_report(nonmiss_y_test, nonmiss_y_pred)
confusion = confusion_matrix(nonmiss_y_test, nonmiss_y_pred)

print("Accuracy:", accuracy)
print("Classification Report:\n", report)
print("Confusion Matrix:\n", confusion)

In [ ]:
test_df = pd.concat([nonmiss_X_test,nonmiss_y_test], axis = 1)

test_df_True = test_df[test_df['danger'] == True].copy()
test_df_False = test_df[test_df['danger'] == False].copy()

# Number of bootstrap samples
num_samples = 1000

# List to store the bootstrap samples
bootstrap_samples = []

# Perform bootstrap sampling
for _ in range(num_samples):
    # Use sample with replacement to create a bootstrap sample
    bootstrap_sample = test_df_False.sample(n=len(test_df_True), replace=True)
    bootstrap_samples.append(bootstrap_sample)

from sklearn.inspection import permutation_importance

permutations_dic = { 0: result_clf}

import random
random_numbers = random.sample(range(0, 1001), 10)
n = 1
for i in random_numbers:
    df = pd.concat([test_df_True,bootstrap_samples[i]], axis=0)
    X_test = df.drop(columns = 'danger').copy()
    y_test = df['danger'].copy()
    result_clf = permutation_importance(
                clf, X_test, y_test, n_repeats=1, random_state=42, n_jobs=-1)
    permutations_dic[i] = result_clf
    n = n + 1 

importances_df = pd.DataFrame()
for i in permutations_dic.keys():
    sorted_importances_idx = permutations_dic[i].importances_mean.argsort()
    importances = pd.DataFrame(
    permutations_dic[i].importances[sorted_importances_idx].T,
    columns=nonmiss_X_test.columns[sorted_importances_idx],
    )
    importances_df = pd.concat([importances_df,importances], axis = 0)


# Calculate the mean importance for each feature across all permutations
mean_importances = importances_df.mean()

# Sort features based on mean importances
sorted_features = mean_importances.sort_values(ascending=True).index

# Plot the boxplot with features ordered by mean importances
plt.figure(figsize=(10, 12))
ax = importances_df[sorted_features].plot.box(vert=False, whis=10)
ax.set_title("Permutation Importances (test set)")
ax.axvline(x=0, color="k", linestyle="--")
ax.set_xlabel("Decrease in accuracy score")
ax.set_yticklabels(['Sea-level air pressure', 'Day type',  'Day name', 'Season',  'Temperature','Wind direction',  'Month',  'Ambient light',
    'Relative Humidity', 'Average wind speed', 'Hour', 'Weather condition', 'Dew Point',  'Error type',   'Road type',   'Province'           
])
ax.figure.tight_layout()
plt.savefig(r".....Fig3.jpg", bbox_inches='tight', dpi = 300)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

quantitative_vars = [ 'Temperature', 'Dew Point', 'Relative Humidity', 'Wind direction',
       'Average wind speed', 'Sea-level air pressure']

categorical_vars = ['Season', 'Month', 'Error type', 'Province', 'Weather condition',
       'Ambient light', 'Day name', 'Day type', 'Hour', 'Road type',
        'Accident in hotspot']

quantitative_corr_matrix = without_missing_feature_df[quantitative_vars].corr()

categorical_corr_matrix = pd.DataFrame(index=categorical_vars, columns=categorical_vars)

for col1 in categorical_vars:
    for col2 in categorical_vars:
        if col1 != col2:
            contingency_table = pd.crosstab(without_missing_feature_df[col1], without_missing_feature_df[col2])
            chi2, _, _, _ = chi2_contingency(contingency_table)
            n = min(contingency_table.shape)
            cramers_v = (chi2 / (without_missing_feature_df.shape[0] * (n - 1)))**0.5
            categorical_corr_matrix.loc[col1, col2] = cramers_v

correlation_data = []

for cat_var in categorical_vars:
    for quant_var in quantitative_vars:
        if without_missing_feature_df[cat_var].nunique() == 2:  
            corr, _ = pointbiserialr(without_missing_feature_df[cat_var], without_missing_feature_df[quant_var])
        else:  
            corr, _ = kendalltau(without_missing_feature_df[cat_var], without_missing_feature_df[quant_var])
        correlation_data.append({'Categorical variables': cat_var, 'Quantitative variables': quant_var, 'Correlation': corr})

correlation_df = pd.DataFrame(correlation_data)


fig, axes = plt.subplots(1, 3, figsize=(32, 8))
plt.subplots_adjust(wspace=0.3)  

sns.heatmap(quantitative_corr_matrix.apply(pd.to_numeric, errors='coerce').round(2), annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5, ax=axes[0])
axes[0].set_title('A) Pearson\'s Correlation - Quantitative Variables', fontsize=16, fontweight='bold')

sns.heatmap(categorical_corr_matrix.astype(float).fillna(1), annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5, ax=axes[1])
axes[1].set_title('B) Cramér\'s V - Categorical Variables', fontsize=16, fontweight='bold')

correlation_matrix = correlation_df.pivot_table(index='Categorical variables', columns='Quantitative variables', values='Correlation', aggfunc='first')
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5, ax=axes[2])
axes[2].set_title('C) Correlation of Categorical vs Quantitative Variables', fontsize=16, fontweight='bold')

plt.savefig(r"............\Sup Fig 1.png", bbox_inches='tight', dpi=300)
plt.show()

Final_features = [#'season' = 'month','day_name' = 'day_type', 'day/night' = hour , 'temp' is associated with a lot of variables
                 'province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']
            
MLs_df = without_missing_feature_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum', 'danger']].copy()
MLs_df.to_pickle(r"...............................\MLs_df.pkl")

# ML models (Best Predictive model)

In [ ]:
#import libraries
import math
import pandas as pd
import numpy as np
import time
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
#Logistic Regression (LR); 
# # K-Nearest Neighbors (KNN); 
#  BalancedRandomForestClassifier (RF); 
# ExtremebGradient Boosting (XGBoost); 
# # Naïve Bias (NB) 
# Support Vector Machine (SVM).
MLs_df = pd.read_pickle(r"..........................\MLs_df.pkl")

metrics = [
    'accuracy', 'balanced_accuracy', 'top_k_accuracy', 'average_precision',
    'neg_brier_score', 'f1', 'f1_micro', 'f1_macro', 'f1_weighted', 'f1_samples',
    'neg_log_loss', 'precision', 'recall', 'jaccard', 'roc_auc',
    'roc_auc_ovr', 'roc_auc_ovo', 'roc_auc_ovr_weighted', 'roc_auc_ovo_weighted',
]

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedBaggingClassifier

# Define features and target variable
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()
X = pd.get_dummies(X, drop_first=False).copy()
logreg_X_train, logreg_X_test, logreg_y_train, logreg_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=2019)

space = dict()
space['base_estimator__solver'] = ['newton-cg', 'saga']
space['base_estimator__penalty'] = ['none', 'l1', 'l2', '‘elasticnet’']
space['base_estimator__C'] = [100, 10, 1.0, 0.1, 0.01]

bbc_logreg = BalancedBaggingClassifier(base_estimator=LogisticRegression(),
                                       sampling_strategy='auto',
                                       replacement=False,
                                       random_state=0,
                                       n_estimators=20)

logreg_search = GridSearchCV(bbc_logreg, space, scoring=['accuracy', 'balanced_accuracy', 'f1', 'f1_micro', 'f1_macro', 'f1_weighted','precision', 'recall','roc_auc'], n_jobs=-1, cv=cv, refit=False, verbose=1)
logreg_X_train, logreg_X_test, logreg_y_train, logreg_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)

logreg_result = logreg_search.fit(logreg_X_train, logreg_y_train)


import pandas as pd

# Extract relevant information from logreg_result.cv_results_
logreg_results_dict = {
    'Model_Number': list(range(1, len(logreg_result.cv_results_['params']) + 1)),
    'Solver': logreg_result.cv_results_['param_base_estimator__solver'],
    'Penalty': logreg_result.cv_results_['param_base_estimator__penalty'],
    'C': logreg_result.cv_results_['param_base_estimator__C'],
    'Accuracy': logreg_result.cv_results_['mean_test_accuracy'],
    'Balanced_Accuracy': logreg_result.cv_results_['mean_test_balanced_accuracy'],
    'F1_Score': logreg_result.cv_results_['mean_test_f1'],
    'F1_Micro': logreg_result.cv_results_['mean_test_f1_micro'],
    'F1_Macro': logreg_result.cv_results_['mean_test_f1_macro'],
    'F1_Weighted': logreg_result.cv_results_['mean_test_f1_weighted'],
    'Precision': logreg_result.cv_results_['mean_test_precision'],
    'Recall': logreg_result.cv_results_['mean_test_recall'],
    'ROC_AUC': logreg_result.cv_results_['mean_test_roc_auc']
}

# Create a DataFrame from the results_dict
logreg_results_df = pd.DataFrame(logreg_results_dict)

# Sort the DataFrame by ROC_AUC in descending order
logreg_results_df = logreg_results_df.sort_values(by='ROC_AUC', ascending=False)

## Choose the best model based on ROC_AUC
#best_model_row = logreg_results_df.iloc[0]
#best_solver = best_model_row['Solver']
#best_penalty = best_model_row['Penalty']
#best_C = best_model_row['C']
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedBaggingClassifier
# Extract the best parameters
best_params = {
    'solver': 'newton-cg',
    'penalty': 'l2',
    'C': 10
}
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()
X = pd.get_dummies(X, drop_first=False).copy()
logreg_X_train, logreg_X_test, logreg_y_train, logreg_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)
# Train the best model on the full training set

logreg_best_model = BalancedBaggingClassifier(
    base_estimator=LogisticRegression(**best_params),
    sampling_strategy='auto',
    replacement=False,
    random_state=0,
    n_estimators=100
)


logreg_best_model.fit(logreg_X_train, logreg_y_train)


print(f"logreg train accuracy: {logreg_best_model.score(logreg_X_train, logreg_y_train):.3f}")
print(f"logreg test accuracy: {logreg_best_model.score(logreg_X_test, logreg_y_test):.3f}")

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

logreg_y_pred = logreg_best_model.predict(logreg_X_test)

# Evaluate the model's performance
logreg_accuracy = accuracy_score(logreg_y_test, logreg_y_pred)
logreg_report = classification_report(logreg_y_test, logreg_y_pred)
logreg_confusion = confusion_matrix(logreg_y_test, logreg_y_pred)

print("Accuracy:", logreg_accuracy)
print("Classification Report:\n", logreg_report)
print("Confusion Matrix:\n", logreg_confusion)

# Make predictions on the test set
logreg_y_pred_proba = logreg_best_model.predict_proba(logreg_X_test)[:, 1]

# Calculate ROC-AUC score
logreg_roc_auc = roc_auc_score(logreg_y_test, logreg_y_pred_proba)
print(f"ROC-AUC Score: {logreg_roc_auc:.4f}")

# Plot ROC curve
logreg_fpr, logreg_tpr, logreg_ = roc_curve(logreg_y_test, logreg_y_pred_proba)
logreg_roc_auc_curve = auc(logreg_fpr, logreg_tpr)


import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, recall_score, precision_score

# Define the number of bootstrap iterations
n_iterations = 1000  # You can adjust this based on your preference

# Lists to store the metric values from each iteration
roc_auc_values = []

for _ in range(n_iterations):
    # Bootstrap sample
    indices = resample(range(len(logreg_y_test)), replace=True)  # Use replace=True to sample with replacement
    y_test_bootstrap = logreg_y_test.iloc[indices]  # Use iloc for integer indexing
    y_pred_proba_bootstrap = logreg_y_pred_proba[indices]
    # Calculate metrics for the bootstrap sample
    roc_auc = roc_auc_score(y_test_bootstrap, y_pred_proba_bootstrap)
    # Store the metric values
    roc_auc_values.append(roc_auc)

# Calculate the 95% confidence intervals
roc_auc_ci = np.percentile(roc_auc_values, [2.5, 97.5])

# Print the results
print(f"ROC-AUC: {logreg_roc_auc:.4f} (95% CI: {roc_auc_ci[0]:.4f} - {roc_auc_ci[1]:.4f})")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedBaggingClassifier

# Define features and target variable
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()
X = pd.get_dummies(X, drop_first=False).copy()

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=2019)

space = dict()
space['base_estimator__n_neighbors'] = [3, 5, 7]  # Adjust the range based on your needs
space['base_estimator__weights'] = ['uniform'] 
space['base_estimator__metric'] = ['euclidean'] 

bbc_knn = BalancedBaggingClassifier(base_estimator=KNeighborsClassifier(),
                                    sampling_strategy='auto',
                                    replacement=False,
                                    random_state=0,
                                    n_estimators=20)

knn_search = GridSearchCV(bbc_knn, space, scoring=['accuracy', 'balanced_accuracy', 'f1', 'f1_micro', 'f1_macro', 'f1_weighted', 'precision', 'recall', 'roc_auc'],
                          n_jobs=-1, cv=cv, refit=False, verbose=1)

knn_X_train, knn_X_test, knn_y_train, knn_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)

knn_result = knn_search.fit(knn_X_train, knn_y_train)


import pandas as pd

# Extract relevant information from logreg_result.cv_results_
knn_results_dict = {
    'Model_Number': list(range(1, len(knn_result.cv_results_['params']) + 1)),
    'Neighbors': knn_result.cv_results_['param_base_estimator__n_neighbors'],
    'Accuracy': knn_result.cv_results_['mean_test_accuracy'],
    'Balanced_Accuracy': knn_result.cv_results_['mean_test_balanced_accuracy'],
    'F1_Score': knn_result.cv_results_['mean_test_f1'],
    'F1_Micro': knn_result.cv_results_['mean_test_f1_micro'],
    'F1_Macro': knn_result.cv_results_['mean_test_f1_macro'],
    'F1_Weighted': knn_result.cv_results_['mean_test_f1_weighted'],
    'Precision': knn_result.cv_results_['mean_test_precision'],
    'Recall': knn_result.cv_results_['mean_test_recall'],
    'ROC_AUC': knn_result.cv_results_['mean_test_roc_auc']
}

# Create a DataFrame from the results_dict
knn_results_df = pd.DataFrame(knn_results_dict)

# Sort the DataFrame by ROC_AUC in descending order
knn_results_df = knn_results_df.sort_values(by='ROC_AUC', ascending=False)
knn_results_df


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedBaggingClassifier

# Extract the best parameters
best_params = {
    'n_neighbors': 7,
     'weights':'uniform',
     'metric':'euclidean'
}

# Define features and target variable
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()
X = pd.get_dummies(X, drop_first=False).copy()
knn_X_train, knn_X_test, knn_y_train, knn_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)


knn_best_model = BalancedBaggingClassifier(base_estimator=KNeighborsClassifier(**best_params),
                                    sampling_strategy='auto',
                                    replacement=False,
                                    random_state=0,
                                    n_estimators=100)


knn_best_model.fit(knn_X_train, knn_y_train)

print(f"KNN train accuracy: {knn_best_model.score(knn_X_train, knn_y_train):.3f}")
print(f"KNN test accuracy: {knn_best_model.score(knn_X_test, knn_y_test):.3f}")

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

knn_y_pred = knn_best_model.predict(knn_X_test)

# Evaluate the model's performance
knn_accuracy = accuracy_score(knn_y_test, knn_y_pred)
knn_report = classification_report(knn_y_test, knn_y_pred)
knn_confusion = confusion_matrix(knn_y_test, knn_y_pred)

print("Accuracy:", knn_accuracy)
print("Classification Report:\n", knn_report)
print("Confusion Matrix:\n", knn_confusion)

# Make predictions on the test set
knn_y_pred_proba = knn_best_model.predict_proba(knn_X_test)[:, 1]

# Calculate ROC-AUC score
knn_roc_auc = roc_auc_score(knn_y_test, knn_y_pred_proba)
print(f"ROC-AUC Score: {knn_roc_auc:.4f}")

# Plot ROC curve
knn_fpr, knn_tpr, knn_ = roc_curve(knn_y_test, knn_y_pred_proba)
knn_roc_auc_curve = auc(knn_fpr, knn_tpr)


import numpy as np

def calculate_metrics(confusion_matrix):
    # Extract values from the confusion matrix
    true_negatives, false_positives, false_negatives, true_positives = confusion_matrix.ravel()

    # Calculate sensitivity (true positive rate)
    sensitivity = true_positives / (true_positives + false_negatives)

    # Calculate specificity (true negative rate)
    specificity = true_negatives / (true_negatives + false_positives)

    # Calculate balanced accuracy
    balanced_accuracy = (sensitivity + specificity) / 2

    return sensitivity, specificity, balanced_accuracy

# Example confusion matrix
conf_matrix = np.array([[139447, 43687],
                       [337, 2526]])

# Calculate metrics
sensitivity, specificity, balanced_accuracy = calculate_metrics(conf_matrix)

# Print the results
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")


import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, recall_score, precision_score

# Define the number of bootstrap iterations
n_iterations = 1000  # You can adjust this based on your preference

# Lists to store the metric values from each iteration
roc_auc_values = []

for _ in range(n_iterations):
    # Bootstrap sample
    indices = resample(range(len(knn_y_test)), replace=True)  # Use replace=True to sample with replacement
    y_test_bootstrap = knn_y_test.iloc[indices]  # Use iloc for integer indexing
    y_pred_proba_bootstrap = knn_y_pred_proba[indices]
    # Calculate metrics for the bootstrap sample
    roc_auc = roc_auc_score(y_test_bootstrap, y_pred_proba_bootstrap)
    # Store the metric values
    roc_auc_values.append(roc_auc)

# Calculate the 95% confidence intervals
roc_auc_ci = np.percentile(roc_auc_values, [2.5, 97.5])

# Print the results
print(f"ROC-AUC: {knn_roc_auc:.4f} (95% CI: {roc_auc_ci[0]:.4f} - {roc_auc_ci[1]:.4f})")

In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, train_test_split

X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()

categorical_columns = ['error_type', 'province', 'final_coco', 'hour', 'road_type']
numerical_columns = ['dwpt', 'rhum','wspd']

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
numerical_pipe = SimpleImputer(strategy='constant', fill_value=-1)

preprocessing = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_pipe, numerical_columns),
    ],
    verbose_feature_names_out=False,
)

RF = Pipeline(
    [
        ("preprocess", preprocessing),
        ("classifier", BalancedRandomForestClassifier(sampling_strategy="auto", replacement=True, random_state=0)),
    ]
)

space = dict()
space['classifier__max_depth'] = [10, 50, 100]
space['classifier__n_estimators'] = [200, 600, 1000, 1400]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=2019)
rf_search = GridSearchCV(RF, space, scoring=['accuracy', 'balanced_accuracy', 'f1', 'f1_micro', 'f1_macro', 'f1_weighted', 'precision', 'recall', 'roc_auc'], n_jobs=-1, cv=cv, refit=False, verbose=1)
rf_X_train, rf_X_test, rf_y_train, rf_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)

rf_result = rf_search.fit(rf_X_train, rf_y_train)


import pandas as pd

# Extract relevant information from logreg_result.cv_results_
rf_results_dict = {
    'Model_Number': list(range(1, len(rf_result.cv_results_['params']) + 1)),
    'Max-Depth': rf_result.cv_results_['param_classifier__max_depth'],
    'N estimator': rf_result.cv_results_['param_classifier__n_estimators'],
    'Accuracy': rf_result.cv_results_['mean_test_accuracy'],
    'Balanced_Accuracy': rf_result.cv_results_['mean_test_balanced_accuracy'],
    'F1_Score': rf_result.cv_results_['mean_test_f1'],
    'F1_Micro': rf_result.cv_results_['mean_test_f1_micro'],
    'F1_Macro': rf_result.cv_results_['mean_test_f1_macro'],
    'F1_Weighted': rf_result.cv_results_['mean_test_f1_weighted'],
    'Precision': rf_result.cv_results_['mean_test_precision'],
    'Recall': rf_result.cv_results_['mean_test_recall'],
    'ROC_AUC': rf_result.cv_results_['mean_test_roc_auc']
}

# Create a DataFrame from the results_dict
rf_results_df = pd.DataFrame(rf_results_dict)

# Sort the DataFrame by ROC_AUC in descending order
rf_results_df = rf_results_df.sort_values(by='ROC_AUC', ascending=False)

best_params = {
    'max_depth': 50,
    'n_estimators': 1400,
}
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()

categorical_columns = ['error_type', 'province', 'final_coco', 'hour', 'road_type']
numerical_columns = ['dwpt', 'rhum', 'wspd']

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
numerical_pipe = SimpleImputer(strategy='constant', fill_value=-1)

preprocessing = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_pipe, numerical_columns),
    ],
    verbose_feature_names_out=False,
)

Best_RF = Pipeline(
    [
        ("preprocess", preprocessing),
        ("classifier", BalancedRandomForestClassifier(**best_params,sampling_strategy="auto", replacement=False, random_state=0)),
    ]
)

Best_RF.fit(rf_X_train, rf_y_train)


print(f"RF train accuracy: {Best_RF.score(rf_X_train, rf_y_train):.3f}")
print(f"RF test accuracy: {Best_RF.score(rf_X_test, rf_y_test):.3f}")

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

rf_y_pred = Best_RF.predict(rf_X_test)

# Evaluate the model's performance
rf_accuracy = accuracy_score(rf_y_test, rf_y_pred)
rf_report = classification_report(rf_y_test, rf_y_pred)
rf_confusion = confusion_matrix(rf_y_test, rf_y_pred)

print("Accuracy:", rf_accuracy)
print("Classification Report:\n", rf_report)
print("Confusion Matrix:\n", rf_confusion)

# Make predictions on the test set
rf_y_pred_proba = Best_RF.predict_proba(rf_X_test)[:, 1]

# Calculate ROC-AUC score
rf_roc_auc = roc_auc_score(rf_y_test, rf_y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Plot ROC curve
rf_fpr, rf_tpr, rf_ = roc_curve(rf_y_test, rf_y_pred_proba)
rf_roc_auc_curve = auc(rf_fpr, rf_tpr)

import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, recall_score, precision_score

# Define the number of bootstrap iterations
n_iterations = 1000  # You can adjust this based on your preference

# Lists to store the metric values from each iteration
roc_auc_values = []

for _ in range(n_iterations):
    # Bootstrap sample
    indices = resample(range(len(rf_y_test)), replace=True)  # Use replace=True to sample with replacement
    y_test_bootstrap = rf_y_test.iloc[indices]  # Use iloc for integer indexing
    y_pred_proba_bootstrap = rf_y_pred_proba[indices]
    # Calculate metrics for the bootstrap sample
    roc_auc = roc_auc_score(y_test_bootstrap, y_pred_proba_bootstrap)
    # Store the metric values
    roc_auc_values.append(roc_auc)

# Calculate the 95% confidence intervals
roc_auc_ci = np.percentile(roc_auc_values, [2.5, 97.5])

# Print the results
print(f"ROC-AUC: {rf_roc_auc:.4f} (95% CI: {roc_auc_ci[0]:.4f} - {roc_auc_ci[1]:.4f})")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, train_test_split
import xgboost as xgb

X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()

categorical_columns = ['error_type', 'province', 'final_coco', 'hour', 'road_type']
numerical_columns = ['dwpt', 'rhum', 'wspd']

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
numerical_pipe = SimpleImputer(strategy='constant', fill_value=-1)

preprocessing = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_pipe, numerical_columns),
    ],
    verbose_feature_names_out=False,
)

xgb_model = Pipeline(
    [
        ("preprocess", preprocessing),
        ("classifier", EasyEnsembleClassifier(estimator=xgb.XGBClassifier(),
            sampling_strategy="auto",
            replacement=True,
            random_state=0
        )),
    ]
)

space = dict()
space['classifier__estimator__max_depth'] = [5, 10, 20]
space['classifier__estimator__n_estimators'] = [10, 100, 1000]
space['classifier__estimator__learning_rate'] = [0.01, 0.1]

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=2019)

xgb_search = GridSearchCV(xgb_model, space, scoring=['accuracy', 'balanced_accuracy', 'f1', 'f1_micro', 'f1_macro', 'f1_weighted', 'precision', 'recall', 'roc_auc'], n_jobs=-1, cv=cv, refit=False, verbose=1)
xgb_X_train, xgb_X_test, xgb_y_train, xgb_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)
xgb_result = xgb_search.fit(xgb_X_train, xgb_y_train)

import pandas as pd

# Extract relevant information from logreg_result.cv_results_
xgb_results_dict = {
    'Model_Number': list(range(1, len(xgb_result.cv_results_['params']) + 1)),
    'lerning rate': xgb_result.cv_results_['param_classifier__estimator__learning_rate'],
    'estimators': xgb_result.cv_results_['param_classifier__estimator__n_estimators'],
    'max depth': xgb_result.cv_results_['param_classifier__estimator__max_depth'],
    'Accuracy': xgb_result.cv_results_['mean_test_accuracy'],
    'Balanced_Accuracy': xgb_result.cv_results_['mean_test_balanced_accuracy'],
    'F1_Score': xgb_result.cv_results_['mean_test_f1'],
    'F1_Micro': xgb_result.cv_results_['mean_test_f1_micro'],
    'F1_Macro': xgb_result.cv_results_['mean_test_f1_macro'],
    'F1_Weighted': xgb_result.cv_results_['mean_test_f1_weighted'],
    'Precision': xgb_result.cv_results_['mean_test_precision'],
    'Recall': xgb_result.cv_results_['mean_test_recall'],
    'ROC_AUC': xgb_result.cv_results_['mean_test_roc_auc']
}

# Create a DataFrame from the results_dict
xgb_results_df = pd.DataFrame(xgb_results_dict)

# Sort the DataFrame by ROC_AUC in descending order
xgb_results_df = xgb_results_df.sort_values(by='ROC_AUC', ascending=False)

best_params = {
    'max_depth': 10,
    'n_estimators': 1000,
    'learning_rate': 0.01
}
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()

categorical_columns = ['error_type', 'province', 'final_coco', 'hour', 'road_type']
numerical_columns = ['dwpt', 'rhum', 'wspd']

categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
numerical_pipe = SimpleImputer(strategy='constant', fill_value=-1)

preprocessing = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_pipe, numerical_columns),
    ],
    verbose_feature_names_out=False,
)

Best_XGB = Pipeline(
    [
        ("preprocess", preprocessing),
        ("classifier", EasyEnsembleClassifier(estimator=xgb.XGBClassifier(**best_params),
            sampling_strategy="auto",
            replacement=True,
            random_state=0
        )),
    ]
)

Best_XGB.fit(xgb_X_train, xgb_y_train)

print(f"XGB train accuracy: {Best_XGB.score(xgb_X_train, xgb_y_train):.3f}")
print(f"XGB test accuracy: {Best_XGB.score(xgb_X_test, xgb_y_test):.3f}")

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

xgb_y_pred = Best_XGB.predict(xgb_X_test)

# Evaluate the model's performance
xgb_accuracy = accuracy_score(xgb_y_test, xgb_y_pred)
xgb_report = classification_report(xgb_y_test, xgb_y_pred)
xgb_confusion = confusion_matrix(xgb_y_test, xgb_y_pred)

print("Accuracy:", xgb_accuracy)
print("Classification Report:\n", xgb_report)
print("Confusion Matrix:\n", xgb_confusion)

# Make predictions on the test set
xgb_y_pred_proba = Best_XGB.predict_proba(xgb_X_test)[:, 1]

# Calculate ROC-AUC score
xgb_roc_auc = roc_auc_score(xgb_y_test, xgb_y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Plot ROC curve
xgb_fpr, xgb_tpr, xgb_ = roc_curve(xgb_y_test, xgb_y_pred_proba)
xgb_roc_auc_curve = auc(xgb_fpr, xgb_tpr)

import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, recall_score, precision_score

# Define the number of bootstrap iterations
n_iterations = 1000  # You can adjust this based on your preference

# Lists to store the metric values from each iteration
roc_auc_values = []

for _ in range(n_iterations):
    # Bootstrap sample
    indices = resample(range(len(xgb_y_test)), replace=True)  # Use replace=True to sample with replacement
    y_test_bootstrap = xgb_y_test.iloc[indices]  # Use iloc for integer indexing
    y_pred_proba_bootstrap = xgb_y_pred_proba[indices]
    # Calculate metrics for the bootstrap sample
    roc_auc = roc_auc_score(y_test_bootstrap, y_pred_proba_bootstrap)
    # Store the metric values
    roc_auc_values.append(roc_auc)

# Calculate the 95% confidence intervals
roc_auc_ci = np.percentile(roc_auc_values, [2.5, 97.5])

# Print the results
print(f"ROC-AUC: {xgb_roc_auc:.4f} (95% CI: {roc_auc_ci[0]:.4f} - {roc_auc_ci[1]:.4f})")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedBaggingClassifier

# Define features and target variable
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()
X = pd.get_dummies(X, drop_first=False).copy()

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=2019)

space = dict()
# Naive Bayes does not have parameters like solver, penalty, and C
# Adjust the parameters accordingly based on your needs
space['base_estimator__alpha'] = [1.0, 0.1, 0.01, 0.001]  # Alpha parameter for MultinomialNB

bbc_nb = BalancedBaggingClassifier(base_estimator=MultinomialNB(),
                                   sampling_strategy='auto',
                                   replacement=False,
                                   random_state=0,
                                   n_estimators=20)

nb_search = GridSearchCV(bbc_nb, space, scoring=['accuracy', 'balanced_accuracy', 'f1', 'f1_micro', 'f1_macro', 'f1_weighted', 'precision', 'recall', 'roc_auc'],
                         n_jobs=-1, cv=cv, refit=False, verbose=1)

nb_X_train, nb_X_test, nb_y_train, nb_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)

nb_result = nb_search.fit(nb_X_train, nb_y_train)

import pandas as pd

# Extracting relevant information from cv_results_
nb_results_dfresults_dict = {
    'Model_Number': list(range(1, len(nb_result.cv_results_['params']) + 1)),
    'Alpha': nb_result.cv_results_['param_base_estimator__alpha'],
    'Accuracy': nb_result.cv_results_['mean_test_accuracy'],
    'Balanced_Accuracy': nb_result.cv_results_['mean_test_balanced_accuracy'],
    'F1_Score': nb_result.cv_results_['mean_test_f1'],
    'F1_Micro': nb_result.cv_results_['mean_test_f1_micro'],
    'F1_Macro': nb_result.cv_results_['mean_test_f1_macro'],
    'F1_Weighted': nb_result.cv_results_['mean_test_f1_weighted'],
    'Precision': nb_result.cv_results_['mean_test_precision'],
    'Recall': nb_result.cv_results_['mean_test_recall'],
    'ROC_AUC': nb_result.cv_results_['mean_test_roc_auc']
}

# Creating a DataFrame
nb_results_df = pd.DataFrame(results_dict)

# Sorting the DataFrame by ROC_AUC in descending order
nb_results_df = nb_results_df.sort_values(by='ROC_AUC', ascending=False)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedBaggingClassifier
# Extract the best parameters
best_params = {
    'alpha': 0.001
}
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()
X = pd.get_dummies(X, drop_first=False).copy()
nb_X_train, nb_X_test, nb_y_train, nb_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)
# Train the best model on the full training set

nb_best_model = BalancedBaggingClassifier(
    base_estimator=MultinomialNB(**best_params),
    sampling_strategy='auto',
    replacement=False,
    random_state=0,
    n_estimators=100
)


nb_best_model.fit(nb_X_train, nb_y_train)

print(f"NB train accuracy: {nb_best_model.score(nb_X_train, nb_y_train):.3f}")
print(f"NB test accuracy: {nb_best_model.score(nb_X_test, nb_y_test):.3f}")

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

nb_y_pred = nb_best_model.predict(nb_X_test)

# Evaluate the model's performance
nb_accuracy = accuracy_score(nb_y_test, nb_y_pred)
nb_report = classification_report(nb_y_test, nb_y_pred)
nb_confusion = confusion_matrix(nb_y_test, nb_y_pred)

print("Accuracy:", nb_accuracy)
print("Classification Report:\n", nb_report)
print("Confusion Matrix:\n", nb_confusion)

# Make predictions on the test set
nb_y_pred_proba = nb_best_model.predict_proba(nb_X_test)[:, 1]

# Calculate ROC-AUC score
nb_roc_auc = roc_auc_score(nb_y_test, nb_y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Plot ROC curve
nb_fpr, nb_tpr, nb_ = roc_curve(nb_y_test, nb_y_pred_proba)
nb_roc_auc_curve = auc(nb_fpr, nb_tpr)

import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, recall_score, precision_score

# Define the number of bootstrap iterations
n_iterations = 1000  # You can adjust this based on your preference

# Lists to store the metric values from each iteration
roc_auc_values = []

for _ in range(n_iterations):
    # Bootstrap sample
    indices = resample(range(len(nb_y_test)), replace=True)  # Use replace=True to sample with replacement
    y_test_bootstrap = nb_y_test.iloc[indices]  # Use iloc for integer indexing
    y_pred_proba_bootstrap = nb_y_pred_proba[indices]
    # Calculate metrics for the bootstrap sample
    roc_auc = roc_auc_score(y_test_bootstrap, y_pred_proba_bootstrap)
    # Store the metric values
    roc_auc_values.append(roc_auc)

# Calculate the 95% confidence intervals
roc_auc_ci = np.percentile(roc_auc_values, [2.5, 97.5])

# Print the results
print(f"ROC-AUC: {nb_roc_auc:.4f} (95% CI: {roc_auc_ci[0]:.4f} - {roc_auc_ci[1]:.4f})")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedBaggingClassifier

# Define features and target variable
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()
X = pd.get_dummies(X, drop_first=False).copy()

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=2019)
space = dict()
space['base_estimator__C'] = [ 0.01, 1, 100]  # Adjust the range based on your needs
space['base_estimator__penalty'] = ['l2'] # Choose the appropriate kernel


bbc_svm = BalancedBaggingClassifier(base_estimator=LinearSVC(dual=False),
                                    sampling_strategy='auto',
                                    replacement=False,
                                    random_state=0,
                                    n_estimators=20)

svm_search = GridSearchCV(bbc_svm, space, scoring=['accuracy', 'balanced_accuracy', 'f1', 'f1_micro', 'f1_macro', 'f1_weighted', 'precision', 'recall', 'roc_auc'],
                          n_jobs=-1, cv=cv, refit=False, verbose=1)

svm_X_train, svm_X_test, svm_y_train, svm_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)

svm_result = svm_search.fit(svm_X_train, svm_y_train)


import pandas as pd

# Extracting relevant information from cv_results_
svm_results_dict = {
    'Model_Number': list(range(1, len(svm_result.cv_results_['params']) + 1)),
    'C': svm_result.cv_results_['param_base_estimator__C'],
    'Accuracy': svm_result.cv_results_['mean_test_accuracy'],
    'Balanced_Accuracy': svm_result.cv_results_['mean_test_balanced_accuracy'],
    'F1_Score': svm_result.cv_results_['mean_test_f1'],
    'F1_Micro': svm_result.cv_results_['mean_test_f1_micro'],
    'F1_Macro': svm_result.cv_results_['mean_test_f1_macro'],
    'F1_Weighted': svm_result.cv_results_['mean_test_f1_weighted'],
    'Precision': svm_result.cv_results_['mean_test_precision'],
    'Recall': svm_result.cv_results_['mean_test_recall'],
    'ROC_AUC': svm_result.cv_results_['mean_test_roc_auc']
}

# Creating a DataFrame
svm_results_df = pd.DataFrame(svm_results_dict)

# Sorting the DataFrame by ROC_AUC in descending order
svm_results_df = svm_results_df.sort_values(by='ROC_AUC', ascending=False)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from imblearn.ensemble import BalancedBaggingClassifier
# Extract the best parameters
best_params = {
    'C': 100
}
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()
X = pd.get_dummies(X, drop_first=False).copy()
svm_X_train, svm_X_test, svm_y_train, svm_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)
# Train the best model on the full training set

svm_best_model = BalancedBaggingClassifier(
    base_estimator=LinearSVC(**best_params, dual=False),
    sampling_strategy='auto',
    replacement=False,
    random_state=0,
    n_estimators=100
)


svm_best_model.fit(svm_X_train,svm_y_train)


print(f"SVM train accuracy: {svm_best_model.score(svm_X_train, svm_y_train):.3f}")
print(f"SVM test accuracy: {svm_best_model.score(svm_X_test, svm_y_test):.3f}")

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

svm_y_pred = svm_best_model.predict(svm_X_test)

# Evaluate the model's performance
svm_accuracy = accuracy_score(svm_y_test, svm_y_pred)
svm_report = classification_report(svm_y_test, svm_y_pred)
svm_confusion = confusion_matrix(svm_y_test, svm_y_pred)

print("Accuracy:", svm_accuracy)
print("Classification Report:\n", svm_report)
print("Confusion Matrix:\n", svm_confusion)

# Make predictions on the test set
svm_y_pred_proba = svm_best_model.predict_proba(svm_X_test)[:, 1]

# Calculate ROC-AUC score
svm_roc_auc = roc_auc_score(svm_y_test, svm_y_pred_proba)
print(f"ROC-AUC Score: {svm_roc_auc:.4f}")

# Plot ROC curve
svm_fpr, svm_tpr, svm_ = roc_curve(svm_y_test, svm_y_pred_proba)
svm_roc_auc_curve = auc(svm_fpr, svm_tpr)

import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, recall_score, precision_score

# Define the number of bootstrap iterations
n_iterations = 1000  # You can adjust this based on your preference

# Lists to store the metric values from each iteration
roc_auc_values = []

for _ in range(n_iterations):
    # Bootstrap sample
    indices = resample(range(len(svm_y_test)), replace=True)  # Use replace=True to sample with replacement
    y_test_bootstrap = svm_y_test.iloc[indices]  # Use iloc for integer indexing
    y_pred_proba_bootstrap = svm_y_pred_proba[indices]
    # Calculate metrics for the bootstrap sample
    roc_auc = roc_auc_score(y_test_bootstrap, y_pred_proba_bootstrap)
    # Store the metric values
    roc_auc_values.append(roc_auc)

# Calculate the 95% confidence intervals
roc_auc_ci = np.percentile(roc_auc_values, [2.5, 97.5])

# Print the results
print(f"ROC-AUC: {svm_roc_auc:.4f} (95% CI: {roc_auc_ci[0]:.4f} - {roc_auc_ci[1]:.4f})")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# Assuming you have computed the ROC curves and AUC values
random_probs = [0 for i in range(len(logreg_y_test))]
p_fpr, p_tpr, _ = roc_curve(logreg_y_test, random_probs)

# Set a more classic and fashionable style
plt.style.use('seaborn-whitegrid')

# Plot ROC curves
plt.figure(figsize=(10, 6))

# Plot individual ROC curves
plt.plot(xgb_fpr, xgb_tpr, linestyle='-', color='blue', label=f'XGBoost (AUC = {xgb_roc_auc_curve:.2f})')
plt.plot(rf_fpr, rf_tpr, linestyle='-', color='red', label=f'Random Forest (AUC = {rf_roc_auc_curve:.2f})')
plt.plot(knn_fpr, knn_tpr, linestyle='-', color='turquoise', label=f'KNN (AUC = {knn_roc_auc_curve:.2f})')
plt.plot(logreg_fpr, logreg_tpr, linestyle='-', color='purple', label=f'Logistic Regression (AUC = {logreg_roc_auc_curve:.2f})')
plt.plot(nb_fpr, nb_tpr, linestyle='-', color='orange', label=f'Naive Bayes (AUC = {nb_roc_auc_curve:.2f})')
plt.plot(svm_fpr, svm_tpr, linestyle='-', color='green', label=f'SVM (AUC = {svm_roc_auc_curve:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='black', label='Random Classifier (AUC = 0.50)', alpha=0.5)


# Set title and labels
plt.title('Receiver Operating Characteristic Curves', fontsize=16)
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)

# Add legend
plt.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.8)

# Add grid for better readability
plt.grid(True, linestyle='--', alpha=0.3)
plt.savefig(r".....................\ROC.png", dpi = 300)
# Show the plot
plt.show()


# SHAP

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, train_test_split
import xgboost as xgb

# Define the features and target
X = MLs_df[['province', 'road_type', 'error_type', 'wspd', 'final_coco', 'hour', 'dwpt', 'rhum']].copy()
y = MLs_df['danger'].copy()
X = pd.get_dummies(X, drop_first=False).copy()

X.rename(columns = {'wspd':'Average wind speed', 'dwpt':'Dew point', 'rhum':'Relative humidity', 'province_Alborz': 'Alborz (province)', 'province_Ardabil': 'Ardabil (province)',
       'province_Azerbaijan, East': 'Azerbaijan, East (province)', 'province_Azerbaijan, West': 'Azerbaijan, West (province)',
       'province_Bushehr': 'Bushehr (province)', 'province_Chahar Mahaal and Bakhtiari': 'Chahar Mahaal and Bakhtiari (province)',
       'province_Fars': 'Fars (province)', 'province_Gilan': 'Gilan (province)', 'province_Golestan': 'Golestan (province)',
       'province_Hamadan': 'Hamadan (province)', 'province_Hormozgan': 'Hormozgan (province)', 'province_Ilam': 'Ilam (province)',
       'province_Isfahan': 'Isfahan (province)', 'province_Kerman': 'Kerman (province)', 'province_Kermanshah': 'Kermanshah (province)',
       'province_Khorasan, North': 'Khorasan, North (province)', 'province_Khorasan, Razavi': 'Khorasan, Razavi (province)',
       'province_Khorasan, South': 'Khorasan, South (province)', 'province_Khuzestan': 'Khuzestan (province)',
       'province_Kohgiluyeh and Boyer-Ahmad': 'Kohgiluyeh and Boyer-Ahmad (province)', 'province_Kordestan': 'Kordestan (province)',
       'province_Markazi': 'Markazi (province)', 'province_Mazandaran': 'Mazandaran (province)', 'province_Qazvin': 'Qazvin (province)',
       'province_Qom': 'Qom (province)', 'province_Semnan': 'Semnan (province)', 'province_Sistan and Baluchistan': 'Sistan and Baluchistan (province)',
       'province_Tehran': 'Tehran (province)', 'province_Yazd': 'Yazd (province)', 'province_Zanjan': 'Zanjan (province)',
       'road_type_minor roads': 'Minor roads', 'road_type_motorway':'Motorway', 'road_type_primary':'Primary roads',
       'road_type_residential':'Residential roads', 'road_type_secondary':'Secondary roads', 'road_type_tertiary':'Tertiary roads',
       'road_type_trunk':'Trunk', 'error_type_0':'Hursh deceleration', 'error_type_1':'Hursh acceleration', 'error_type_2':'Hursh turning',
       'error_type_5':'Over speed', 'error_type_6':'Fatigue', 'final_coco_Clear':'Clear weather', 'final_coco_Cloudy':'Cloudy weather',
       'final_coco_Foggy':'Foggy weather', 'final_coco_Rainy':'Rainy weather', 'final_coco_Snowy':'Snowy weather',
       'final_coco_Storm': 'Storm weather', 'hour_0': 'hour = 0', 'hour_1': 'hour = 1', 'hour_10': 'hour = 10', 'hour_11': 'hour = 11', 'hour_12': 'hour = 12',
       'hour_13': 'hour = 13', 'hour_14': 'hour = 14', 'hour_15': 'hour = 15', 'hour_16': 'hour = 16', 'hour_17': 'hour = 17', 'hour_18': 'hour = 18',
       'hour_19': 'hour = 19', 'hour_2': 'hour = 2', 'hour_20': 'hour = 20', 'hour_21': 'hour = 21', 'hour_22': 'hour = 22', 'hour_23': 'hour = 23',
       'hour_3': 'hour = 3', 'hour_4': 'hour = 4', 'hour_5': 'hour = 5', 'hour_6': 'hour = 6', 'hour_7': 'hour = 7', 'hour_8': 'hour = 8', 'hour_9': 'hour = 9'}, inplace=True)

# Define the XGBClassifier with the best parameters
best_params = {
    'max_depth': 10,
    'n_estimators': 1000,
    'learning_rate': 0.01
}

xgb_classifier = xgb.XGBClassifier(**best_params)

# Define the EasyEnsembleClassifier
easy_ensemble_classifier = EasyEnsembleClassifier(
    estimator=xgb_classifier,
    sampling_strategy="auto",
    replacement=True,
    random_state=0
)
esay_X_train, easy_X_test, easy_y_train, easy_y_test = train_test_split(X, y, test_size=0.3, random_state=2019)
# Fit the EasyEnsembleClassifier directly without a pipeline
easy_ensemble_classifier.fit(esay_X_train, easy_y_train)

print(f"XGB train accuracy: {easy_ensemble_classifier.score(esay_X_train, easy_y_train):.3f}")
print(f"XGB test accuracy: {easy_ensemble_classifier.score(easy_X_test, easy_y_test):.3f}")

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc

easy_y_pred = easy_ensemble_classifier.predict(easy_X_test)

# Evaluate the model's performance
easy_accuracy = accuracy_score(easy_y_test, easy_y_pred)
easy_report = classification_report(easy_y_test, easy_y_pred)
easy_confusion = confusion_matrix(easy_y_test, easy_y_pred)

print("Accuracy:", easy_accuracy)
print("Classification Report:\n", easy_report)
print("Confusion Matrix:\n", easy_confusion)

# Make predictions on the test set
easy_y_pred_proba = easy_ensemble_classifier.predict_proba(easy_X_test)[:, 1]

# Calculate ROC-AUC score
easy_roc_auc = roc_auc_score(easy_y_test, easy_y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Plot ROC curve
easy_fpr, easy_tpr, xgb_ = roc_curve(easy_y_test, easy_y_pred_proba)
easy_roc_auc_curve = auc(easy_fpr, easy_tpr)

In [ ]:
import shap
basemodel_dict = dict()
explainer_dict = dict()
shap_values_dict = dict()
for i in range(0,2):
    basemodel_dict[i] = easy_ensemble_classifier.estimators_[i].steps[-1][1]
    explainer_dict[i] = shap.TreeExplainer(basemodel_dict[i])
    shap_values_dict[i] = explainer_dict[i](easy_X_test)

import pickle

# Specify the file path
file_path = r"..........................\shaply01.pkl"

# Save the dictionary to a file using pickle
with open(file_path, 'wb') as file:
    pickle.dump(shap_values_dict, file)

In [ ]:
import shap
import pickle

file_path1 = r"..............\shaply01.pkl"
file_path2 = r".............\shaply02.pkl"
file_path3 = r"...............\shaply03.pkl"
file_path4 = r"...................\shaply04.pkl"
file_path5 = r"..................\shaply05.pkl"

with open(file_path1, 'rb') as file:
    shaply_data1 = pickle.load(file)

with open(file_path2, 'rb') as file:
    shaply_data2 = pickle.load(file)

with open(file_path3, 'rb') as file:
    shaply_data3 = pickle.load(file)

with open(file_path4, 'rb') as file:
    shaply_data4 = pickle.load(file)

with open(file_path5, 'rb') as file:
    shaply_data5 = pickle.load(file)

with open(file_path1, 'rb') as file:
    shaply_for_visu = pickle.load(file)

shaply_total = {**shaply_data1, **shaply_data2, **shaply_data3, **shaply_data4, **shaply_data5}

In [ ]:
shap_total_array = (shaply_total[0].values +  shaply_total[1].values +  shaply_total[2].values +  shaply_total[3].values +  shaply_total[4].values +  shaply_total[5].values +
                     shaply_total[6].values +  shaply_total[7].values +  shaply_total[8].values +  shaply_total[9].values)/10

shaply_for_visu[0].values = shap_total_array

In [ ]:
shap.plots.beeswarm(shaply_for_visu[0], max_display=100,show=False)
plt.savefig(r"......................\complete_beeswarm.png", bbox_inches='tight', dpi=300)

shap.plots.bar(shaply_for_visu[0], max_display=70,show=False)
plt.savefig(r"........................\complete_bar.png", bbox_inches='tight', dpi=300)

shap.plots.beeswarm(shaply_for_visu[0], max_display=17,show=False)
plt.savefig(r"........................\main_beeswarm.tiff", bbox_inches='tight', dpi=300)

shap.plots.bar(shaply_for_visu[0], max_display=17,show=False)
plt.savefig(r"........................\main_bar.tiff", bbox_inches='tight', dpi=300)

shap.plots.beeswarm(shaply_for_visu[0].abs, color="shap_red", max_display = 17,show=False)
plt.savefig(r"...........................\main_beeswarm_abs.tiff", bbox_inches='tight', dpi=300)